# 01c — Cohorts, 2025 Q3, and ID matching

Loads `mc_all_quarters.parquet` (from `01b_all_quarters`), gives every business one ID across all quarters,
recovers the missing 2025 Q3 IDs, and saves the result as `mc_matched.parquet` for the next notebooks.

**Rules for this notebook**
- The raw files are never changed. Anything cleaned is saved as a new file.
- Nothing is dropped. Problem rows get a flag column instead.
- Data never goes into Git. Clear all outputs before committing.

**The preprocessing notebooks, in order** (each one loads what the one before saved):
1. `01a_id_check`: did the ID change, or only its column name? (stands alone)
2. `01b_all_quarters`: reads the 30 Excel files (2023 Q1 – 2026 Q3) once and saves `mc_all_quarters.parquet`
3. `01c_id_matching`: one ID per business, saves `mc_matched.parquet`
4. `01d_survival_label`: the one-year label, saves `labels.parquet`
5. `01e_ssic_classification`: SSIC from the name, saves `ssic_by_business.csv`, `business_names.csv`, `business_city.csv`
6. `01f_old_data_branches`: old data (2023 Q1 – 2025 Q2), every branch joined to its main (stands alone)

### Cell 1 — Setup
Finds the data. On Colab it downloads the raw files from the team's Google Drive folder (temporary copies,
gone when the session ends). On a laptop it reads the local copy in `DATA_DIR`.

The results that the next notebook loads go in `SAVED_DIR`: the folder **saved** inside the team's shared Drive
folder **Darb DataSets**. Everyone loads the same results, and they are still there in a new Colab session.
On Colab: add a shortcut to Darb DataSets in your My Drive first (Drive → right-click the folder → Organize →
Add shortcut → My Drive). Colab asks once to connect Drive.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    !pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
    from google.colab import drive
    drive.mount('/content/drive')
    TEAM_DIR = Path('/content/drive/MyDrive/Darb DataSets')     # the team's shared folder (a shortcut in My Drive)
    if not TEAM_DIR.exists():
        raise FileNotFoundError('Darb DataSets is not in your My Drive: add a shortcut to it (see the text above), then run this cell again.')
    SAVED_DIR = TEAM_DIR / 'saved'                              # shared with the team, kept between Colab sessions
else:
    # local copy of the Drive folder "Darb DataSets" (outside Git and OneDrive)
    DATA_DIR = Path(os.environ.get('DARB_DATA', r'C:\Users\Nasser Alyahya\darb-data'))
    SAVED_DIR = DATA_DIR / 'saved'

MC_DIR = DATA_DIR / 'MC'
SAVED_DIR.mkdir(parents=True, exist_ok=True)
print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR, '| saved results:', SAVED_DIR)

### Load — the table from `01b_all_quarters`

In [ ]:
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PATH = SAVED_DIR / 'mc_all_quarters.parquet'
if not PATH.exists():
    raise FileNotFoundError(f'{PATH} not found: run 01b_all_quarters first.')
# Text is loaded in the same compact format as in 01b_all_quarters (pd.read_parquet would use about twice the memory)
TEXT = pd.StringDtype('pyarrow')
mc = pq.read_table(PATH).to_pandas(types_mapper={pa.string(): TEXT, pa.large_string(): TEXT}.get)
print(f'Loaded {PATH.name}: {len(mc):,} rows | memory used: {mc.memory_usage(deep=True).sum() / 1e9:.1f} GB')

In [ ]:
QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']

### Cell 11 — How big is each cohort? (only counts)
A cohort is all main records that started in the same quarter. For each cohort: how many businesses we see,
whether the quarter it started in is trustworthy, and whether the quarter one year later (needed for the
1-year label) is trustworthy.

In [ ]:
# Quarters whose file is complete enough to say "this business is gone" (from 01b_all_quarters Cell 10)
TRUSTED = ['2025Q2', '2025Q4', '2026Q1', '2026Q2', '2026Q3']

mains = mc[(mc['record_type'] == 'رئيسي') & mc['id'].notna()]
cohort = pd.to_datetime(mains['start_date'].str[:10], errors='coerce').dt.to_period('Q').astype(str)
mains = mains.assign(cohort=cohort)            # cohort = the quarter the business started in
businesses = mains.drop_duplicates('id')       # each business once

rows = []
for quarter in QUARTERS:
    one_year_later = str(pd.Period(quarter, 'Q') + 4)
    rows.append({
        'cohort (start quarter)': quarter,
        'businesses (seen in any quarter)': int((businesses['cohort'] == quarter).sum()),
        'seen in their own quarter': int(mains.loc[(mains['quarter'] == quarter) & (mains['cohort'] == quarter), 'id'].nunique()),
        'own quarter trustworthy': 'yes' if quarter in TRUSTED else 'no',
        '1-year check in': one_year_later,
        '1-year quarter trustworthy': 'yes' if one_year_later in TRUSTED else 'no (or no data yet)',
    })
print(pd.DataFrame(rows).set_index('cohort (start quarter)').to_string())
print()
print(f'Businesses that started before 2024 Q3: {int((businesses["cohort"] < "2024Q3").sum()):,}')
print(f'Businesses with no readable start date: {int((businesses["cohort"] == "NaT").sum()):,}')


### Cell 12 — What is 2025 Q3 PART1? (only counts)
The 109,541 rows with no ID: which start dates they cover, their capital and entity type, and whether each
row can be found in 2025 Q2 or 2025 Q4 by the same name, start date and city (if so, its ID can be recovered).

In [ ]:
p1 = mc[(mc['quarter'] == '2025Q3') & mc['id'].isna()]
p1_day = p1['start_date'].str[:10]

print(f'2025 Q3 PART1: {len(p1):,} rows (no ID)')
print(f'Start dates from {p1_day.min()} to {p1_day.max()}')
print()
print('Rows per start month:')
print(p1_day.str[:7].value_counts().sort_index().to_string())
print()
print('Capital (most common values):')
print(p1['capital'].value_counts().head(5).to_string())
print()
print('Entity type:')
print(p1['entity_type'].value_counts().to_string())

# Can each row be found in another quarter by name + start date + city? (If yes, its ID can be recovered.)
left = p1.assign(start_day=p1_day)[['name', 'start_day', 'city']].reset_index()

def find_in(quarter):
    other = mc.loc[(mc['quarter'] == quarter) & mc['id'].notna(), ['id', 'name', 'start_date', 'city']]
    other = other.assign(start_day=other['start_date'].str[:10])[['id', 'name', 'start_day', 'city']]
    found = left.merge(other, on=['name', 'start_day', 'city'], how='left')
    ids_per_row = found.groupby('index')['id'].nunique()
    print(f'Found in {quarter}: exactly one ID: {(ids_per_row == 1).sum():,} | '
          f'more than one: {(ids_per_row > 1).sum():,} | not found: {(ids_per_row == 0).sum():,}')

print()
find_in('2025Q2')
find_in('2025Q4')


### Cell 13 — ID matching (adds two columns, deletes nothing)
The agreed rule:
1. Same ID = same business.
2. An old ID that disappears and never comes back is linked to a new ID only if: same name once the legal
   form is removed, same city, the new start date is later, and there is exactly one candidate **both ways**
   (the new ID is not claimed by another old ID either). Only between trustworthy quarters.
3. Unsure cases are never linked: flagged `ambiguous_not_linked`.

Adds `entity_id` (one ID per business across all quarters) and `link_flag`
(`same_id`, `linked_new_id`, `ambiguous_not_linked`, `duplicate_across_parts`, `no_id`).

After a restart, run only Cell 1, the Load cell and this cell (Cells 11 and 12 only count).

In [ ]:
import gc
import re
import numpy as np

# Free memory: the test tables of Part 1 and the checks we no longer need
for name in ['q2', 'q4', 'pairs', 'missing', 'new_q4', 'strict', 'loose', 'p1', 'p1_day', 'left',
             'mains', 'businesses', 'cohort', 'present', 'start_day', 'repeated', 'health', 'links']:
    globals().pop(name, None)
gc.collect()

QUARTERS = ['2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1', '2026Q2', '2026Q3']
TRUSTED = ['2025Q2', '2025Q4', '2026Q1', '2026Q2', '2026Q3']     # complete quarters (01b_all_quarters Cell 10)

def core_name(name):
    """The name without its legal form and spelling differences, for comparing only:
    'شركة الوليد صالح الحربي المحدودة' and 'مؤسسة الوليد صالح الحربي' both become 'الوليد صالح الحربي'."""
    t = str(name)
    t = re.sub('[إأآ]', 'ا', t).replace('ة', 'ه').replace('ى', 'ي')
    t = re.sub(r'\b(شركه|مؤسسه|المحدوده|محدوده|ذات مسؤوليه|شخص واحد|مساهمه|مبسطه|فرع)\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

is_main = mc['record_type'] == 'رئيسي'
has_id = mc['id'].notna()

# The IDs of each quarter, each ID once
ids_in = {q: pd.Index(mc.loc[(mc['quarter'] == q) & has_id, 'id'].unique()) for q in QUARTERS}

def ids_in_any(quarters):
    out = ids_in[quarters[0]]
    for q in quarters[1:]:
        out = out.union(ids_in[q])
    return out

def main_rows(quarter, ids):
    rows = mc.loc[(mc['quarter'] == quarter) & is_main, ['id', 'name', 'city', 'start_date']]
    rows = rows[rows['id'].isin(ids)].drop_duplicates('id')
    return rows.assign(day=rows['start_date'].str[:10], core=rows['name'].map(core_name)).drop(columns='start_date')

# Links old ID -> new ID, only between trustworthy quarters that follow each other
found_links, ambiguous, summary = [], set(), []
for i, (old_q, new_q) in enumerate(zip(TRUSTED, TRUSTED[1:])):
    vanished = ids_in[old_q].difference(ids_in_any(TRUSTED[i + 1:]))                     # gone, and never back
    appeared = ids_in[new_q].difference(ids_in_any(QUARTERS[:QUARTERS.index(old_q) + 1]))  # new, never seen before
    old, new = main_rows(old_q, vanished), main_rows(new_q, appeared)

    pairs = old.merge(new, on=['core', 'city'], suffixes=('_old', '_new'))
    pairs = pairs[(pairs['day_new'] > pairs['day_old']).fillna(False)]   # the new registration is later (empty date = no)
    per_old = pairs.groupby('id_old')['id_new'].transform('nunique')
    per_new = pairs.groupby('id_new')['id_old'].transform('nunique')
    good = pairs[(per_old == 1) & (per_new == 1)]                         # one candidate, both ways
    unsure = set(pairs['id_old']) - set(good['id_old'])
    ambiguous |= unsure
    found_links.append(good.assign(old_quarter=old_q, new_quarter=new_q))
    summary.append({'from': old_q, 'to': new_q, 'gone (main)': len(old), 'new (main)': len(new),
                    'linked': len(good), 'unsure, not linked': len(unsure)})
    del vanished, appeared, old, new, pairs

links = pd.concat(found_links, ignore_index=True)
del ids_in, found_links
gc.collect()

# One entity_id per business: a linked new ID takes the entity_id of the old one (also through chains).
# Only the linked rows change; every other row keeps its own ID.
entity_of = {}
for old_id, new_id in zip(links['id_old'], links['id_new']):
    entity_of[new_id] = entity_of.get(old_id, old_id)
mc['entity_id'] = mc['id']
linked_rows = mc['id'].isin(list(entity_of)).to_numpy()
mc.loc[linked_rows, 'entity_id'] = mc.loc[linked_rows, 'id'].map(entity_of)

# How each row was linked. Nothing is deleted.
FLAGS = ['same_id', 'linked_new_id', 'ambiguous_not_linked', 'duplicate_across_parts', 'no_id']
codes = np.zeros(len(mc), dtype=np.int8)                       # 0 = same_id
codes[linked_rows] = 1
codes[mc['id'].isin(list(ambiguous)).to_numpy()] = 2
for quarter in QUARTERS:                                       # the 2nd copy of an ID inside one quarter
    in_quarter = np.flatnonzero((mc['quarter'] == quarter).to_numpy())
    ids = mc['id'].iloc[in_quarter]
    codes[in_quarter[(ids.notna() & ids.duplicated()).to_numpy()]] = 3
codes[(~has_id).to_numpy()] = 4
mc['link_flag'] = pd.Categorical.from_codes(codes, FLAGS)
del codes
gc.collect()

print(pd.DataFrame(summary).set_index('from').to_string())
print()
print('Rows per link_flag:')
print(mc['link_flag'].value_counts().to_string())
print()
print(f'Different IDs: {mc["id"].nunique():,} | different businesses (entity_id): {mc["entity_id"].nunique():,}')
print(f'Memory used by the data: {mc.memory_usage(deep=True).sum() / 1e9:.1f} GB')
print()
print('10 random links:')
links[['old_quarter', 'id_old', 'name_old', 'day_old', 'id_new', 'name_new', 'day_new', 'city']].sample(min(10, len(links)), random_state=1)


### Cell 13b — 2025 Q3: recover the missing IDs (adds two columns, deletes nothing)
2025 Q3 PART1 has no ID column. Most of its rows are businesses already in 2025 Q2 (Cell 12).

**Rule:** a row gets the ID of the 2025 Q2 row with the **same name + start date + city**, only if there is
**exactly one** such ID. More than one, or none: no ID (never guessed).
**Safety check:** if the recovered ID is already in 2025 Q3, or two rows would get the same ID, those rows are cancelled.

Adds `recovered_id` and `id_source` (`original`, `recovered_from_2025Q2`, `not_recovered_many_matches`,
`not_recovered_not_found`, `not_recovered_conflict`). The column `id` is **not** changed.
Recovered rows get their `entity_id` and `link_flag = recovered_id`.
The survival labels do not use these rows yet (that is the next step).

Needs Cell 1, the Load cell and Cell 13 run first.

In [ ]:
import gc
import numpy as np

if 'entity_of' not in globals():
    raise NameError('Run Cell 13 first (this cell needs entity_id and link_flag).')

# The 2025 Q3 PART1 rows with no ID, and their key: name + start date + city
no_id = mc.index[(mc['quarter'] == '2025Q3') & mc['id'].isna()]
p1 = pd.DataFrame({'row': no_id,
                   'name': mc.loc[no_id, 'name'].to_numpy(),
                   'day': mc.loc[no_id, 'start_date'].str[:10].to_numpy(),
                   'city': mc.loc[no_id, 'city'].astype(str).to_numpy()})
has_key = p1['name'].notna() & p1['day'].notna() & (p1['city'] != 'nan')

# The same key for every 2025 Q2 row that has an ID (Q2 is complete and is the quarter just before)
in_q2 = (mc['quarter'] == '2025Q2') & mc['id'].notna()
q2 = pd.DataFrame({'name': mc.loc[in_q2, 'name'].to_numpy(),
                   'day': mc.loc[in_q2, 'start_date'].str[:10].to_numpy(),
                   'city': mc.loc[in_q2, 'city'].astype(str).to_numpy(),
                   'q2_id': mc.loc[in_q2, 'id'].to_numpy()})
q2 = q2.dropna(subset=['name', 'day']).drop_duplicates()

# How many different Q2 IDs share each row's key
found = p1[has_key].merge(q2, on=['name', 'day', 'city'], how='left')
ids_per_row = found.groupby('row')['q2_id'].nunique().reindex(p1['row'], fill_value=0)
one_match = ids_per_row.index[ids_per_row == 1]
many_matches = ids_per_row.index[ids_per_row > 1]
recovered = found.dropna(subset=['q2_id']).drop_duplicates('row').set_index('row')['q2_id'].loc[one_match]

# Safety check: no guessing when the recovered ID is already in 2025 Q3, or two rows get the same ID
q3_ids = pd.Index(mc.loc[(mc['quarter'] == '2025Q3') & mc['id'].notna(), 'id'].unique())
conflict = recovered.index[recovered.duplicated(keep=False) | recovered.isin(q3_ids)]
recovered = recovered.drop(conflict)
del found, q2, q3_ids
gc.collect()

# New column id_source (for every row). The column id itself is NOT changed.
SOURCES = ['original', 'recovered_from_2025Q2', 'not_recovered_many_matches',
           'not_recovered_not_found', 'not_recovered_conflict']
codes = np.zeros(len(mc), dtype=np.int8)                          # 0 = original
codes[mc.index.get_indexer(no_id)] = 3                            # start as "not found"
codes[mc.index.get_indexer(many_matches)] = 2
codes[mc.index.get_indexer(recovered.index)] = 1
codes[mc.index.get_indexer(conflict)] = 4
mc['id_source'] = pd.Categorical.from_codes(codes, SOURCES)
del codes

# New column recovered_id, and the recovered rows join their business (entity_id, through Cell 13's links)
mc['recovered_id'] = pd.Series(pd.NA, index=mc.index, dtype='string[pyarrow]')
mc.loc[recovered.index, 'recovered_id'] = recovered.to_numpy()
mc.loc[recovered.index, 'entity_id'] = [entity_of.get(i, i) for i in recovered]
if 'recovered_id' not in mc['link_flag'].cat.categories:
    mc['link_flag'] = mc['link_flag'].cat.add_categories(['recovered_id'])
mc.loc[recovered.index, 'link_flag'] = 'recovered_id'
gc.collect()

# Results
print(f'2025 Q3 rows with no ID: {len(no_id):,}  (rows with an empty name, date or city: {(~has_key).sum():,})')
print()
print('id_source of these rows:')
print(mc.loc[no_id, 'id_source'].value_counts().to_string())
print()
print(f'Conflicts cancelled (ID already in 2025 Q3, or the same ID for two rows): {len(conflict):,}')
print('Recovered rows by record type:')
print(mc.loc[recovered.index, 'record_type'].value_counts().to_string())
print()
print(f'Rows per link_flag now:')
print(mc['link_flag'].value_counts().to_string())

# 10 random recovered rows, with the name the same ID has in 2025 Q4 (a check by eye)
in_q4 = mc.loc[(mc['quarter'] == '2025Q4') & mc['id'].isin(recovered.to_numpy()), ['id', 'name']]
name_in_q4 = in_q4.drop_duplicates('id').set_index('id')['name']
examples = mc.loc[recovered.index, ['name', 'start_date', 'city', 'recovered_id']].sample(min(10, len(recovered)), random_state=1)
examples['start_date'] = examples['start_date'].str[:10]
examples['name in 2025 Q4 (same ID)'] = examples['recovered_id'].map(name_in_q4).fillna('(not in 2025 Q4)')
print()
print('10 random recovered rows:')
print(examples.to_string(index=False))
del p1, in_q4, name_in_q4
gc.collect()


### Cell 13c — Record type for the recovered rows (fills empty values only, deletes nothing)
2025 Q3 PART1 has no record type column, so the rows recovered in Cell 13b do not say if they are a main
record (رئيسي) or a branch (فرعي). The labels use main records only, so without this they would be ignored.

**Rule:** copy the record type of the **same ID in 2025 Q2**. Only empty values are filled.
Adds `record_type_source` (`original`, `copied_from_2025Q2`, `missing`).
Also counts rows whose record type is the column title "نوع السجل" (a header row read as data). Cell 8 now
removes such rows, so this should print 0.

Needs Cell 1, the Load cell, 13 and 13b run first. Safe to run again.

In [ ]:
import gc
import numpy as np

if 'id_source' not in mc.columns:
    raise NameError('Run Cell 13b first (this cell needs recovered_id and id_source).')

# The recovered rows (Cell 13b) have no record type: 2025 Q3 PART1 has no such column
recovered_rows = mc.index[(mc['id_source'] == 'recovered_from_2025Q2') & mc['record_type'].isna()]

# The record type of the same ID in 2025 Q2 (first row per ID)
in_q2 = (mc['quarter'] == '2025Q2') & mc['id'].notna() & mc['record_type'].notna()
type_in_q2 = (mc.loc[in_q2, ['id', 'record_type']].drop_duplicates('id')
              .set_index('id')['record_type'].astype(str))
copied = mc.loc[recovered_rows, 'recovered_id'].map(type_in_q2).dropna()

# Fill only the empty record types, and say where each record type came from
mc.loc[copied.index, 'record_type'] = copied.to_numpy()
SOURCES = ['original', 'copied_from_2025Q2', 'missing']
codes = np.where(mc['record_type'].isna().to_numpy(), 2, 0).astype(np.int8)
if 'record_type_source' in mc.columns:                       # running again keeps earlier copies marked
    codes[(mc['record_type_source'] == 'copied_from_2025Q2').to_numpy()] = 1
codes[mc.index.get_indexer(copied.index)] = 1
mc['record_type_source'] = pd.Categorical.from_codes(codes, SOURCES)
del codes, type_in_q2
gc.collect()

print(f'Recovered rows with no record type: {len(recovered_rows):,} | record type copied from 2025 Q2: {len(copied):,}')
print()
print('Copied record types:')
print(copied.value_counts().to_string())
print()
print('record_type_source (all rows):')
print(mc['record_type_source'].value_counts().to_string())
print()
print('Rows still with no record type, per quarter:')
print(mc.loc[mc['record_type'].isna(), 'quarter'].value_counts().sort_index().to_string())
print()
header_rows = mc['record_type'] == 'نوع السجل'
print(f'Rows whose record type is the column title "نوع السجل" (a header row read as data): {int(header_rows.sum()):,}')
if header_rows.any():
    print(mc.loc[header_rows, ['quarter', 'source_file', 'id', 'name']].head(10).to_string())


### Save — the matched table for the next notebooks
Saves `mc` with its new columns (`entity_id`, `link_flag`, `id_source`, `recovered_id`, `record_type_source`) to
`mc_matched.parquet` in `SAVED_DIR`. `01d_survival_label` and `01e_ssic_classification` load it.

In [ ]:
OUT = SAVED_DIR / 'mc_matched.parquet'
mc.to_parquet(OUT, index=False)
print(f'Saved {len(mc):,} rows to {OUT} ({OUT.stat().st_size / 1e6:,.0f} MB)')